In [1]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from langchain_ollama.embeddings import OllamaEmbeddings
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import PydanticOutputParser

from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_neo4j import Neo4jGraph, Neo4jVector

/var/folders/xc/lny8rntd7nn0g08bxqjc_s000000gn/T/ipykernel_58450/400796529.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/var/folders/xc/lny8rntd7nn0g08bxqjc_s000000gn/T/ipykernel_58450/400796529.py:11: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.graph_transformers import LLMGraphTransformer


In [2]:
load_dotenv()

True

In [3]:
# temperature=0 ensures deterministic entity and relationship extraction
llm = ChatOllama(model="qwen3.5:9b", temperature=0)
embeddings = OllamaEmbeddings(model="embeddinggemma")

In [4]:
# PyPDFLoader yields one Document per page
loader = PyPDFLoader("data/elon_musk.pdf")
pages = loader.load()

for i, p in enumerate(pages):
    print(f"Page {i + 1}: {len(p.page_content)} chars")

Page 1: 2343 chars
Page 2: 1192 chars


In [5]:
# smaller chunks give the LLM tighter context for entity extraction
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(pages)

print(f"{len(chunks)} chunks created")

14 chunks created


In [6]:
graph = Neo4jGraph(
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
    database="f9f44ab1"
)

In [7]:
graph_transformer = LLMGraphTransformer(llm=llm, ignore_tool_usage=True)

In [ ]:
graph_docs = graph_transformer.convert_to_graph_documents(chunks)

print(f"{len(graph_docs)} graph documents extracted")

# spot-check the first extraction
print("Nodes:", [n.id for n in graph_docs[0].nodes])
print("Rels: ", [(r.source.id, r.type, r.target.id) for r in graph_docs[0].relationships])

graph_docs = graph_transformer.convert_to_graph_documents(chunks)

print(f"{len(graph_docs)} graph documents extracted")

# spot-check the first extraction
print("Nodes:", [n.id for n in graph_docs[0].nodes])
print("Rels: ", [(r.source.id, r.type, r.target.id) for r in graph_docs[0].relationships])

In [10]:
# include_source=True links each entity node back to its source Document node,
# which is required for Neo4jVector.from_existing_graph in the next cell
graph.add_graph_documents(
    graph_docs,
    include_source=True,
    baseEntityLabel=True
)
print("Graph stored in Neo4J")

[#CBF3]  _: <CONNECTION> error: Failed to read from defunct connection ResolvedIPv6Address(('64:ff9b::227c:a9ab', 7687, 0, 0)) (ResolvedIPv6Address(('64:ff9b::227c:a9ab', 7687, 0, 0))): ConnectionResetError(54, 'Connection reset by peer')
[#CBF4]  _: <CONNECTION> error: Failed to read from defunct connection IPv4Address(('p-mt-2e58c6e6c448-26-0189.production-orch-0703.neo4j.io', 7687)) (ResolvedIPv6Address(('64:ff9b::227c:a9ab', 7687, 0, 0))): ConnectionResetError(54, 'Connection reset by peer')
Transaction failed and will be retried in 1.1066648283065177s (Failed to read from defunct connection IPv4Address(('p-mt-2e58c6e6c448-26-0189.production-orch-0703.neo4j.io', 7687)) (ResolvedIPv6Address(('64:ff9b::227c:a9ab', 7687, 0, 0))))


Graph stored in Neo4J


In [12]:
# create a vector index over the Document nodes stored above
vector_index = Neo4jVector.from_existing_graph(
    embedding=embeddings,
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
    index_name="elon_musk_chunks",
    database="f9f44ab1",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
)
print("Vector index created")

Vector index created


In [13]:
# verify what landed in Neo4J
node_counts = graph.query(
    "MATCH (n) RETURN labels(n) AS label, count(n) AS count ORDER BY count DESC"
)
rel_counts = graph.query(
    "MATCH ()-[r]->() RETURN type(r) AS type, count(r) AS count ORDER BY count DESC"
)
print("Nodes:")
for r in node_counts:
    print(" ", r)
print("Relationships:")
for r in rel_counts:
    print(" ", r)

Nodes:
  {'label': ['__Entity__', 'Occupation'], 'count': 7}
  {'label': ['__Entity__', 'Person'], 'count': 5}
  {'label': ['__Entity__', 'Location'], 'count': 3}
  {'label': ['Document'], 'count': 2}
  {'label': ['__Entity__', 'Date'], 'count': 1}
  {'label': ['__Entity__', 'Nationality'], 'count': 1}
  {'label': ['__Entity__', 'Title'], 'count': 1}
  {'label': ['__Entity__', 'Country'], 'count': 1}
Relationships:
  {'type': 'MENTIONS', 'count': 20}
  {'type': 'OCCUPATION', 'count': 5}
  {'type': 'IS_OCCUPATION', 'count': 2}
  {'type': 'LOCATED_IN', 'count': 2}
  {'type': 'BORN_ON', 'count': 1}
  {'type': 'BORN_IN', 'count': 1}
  {'type': 'HAS_NATIONALITY', 'count': 1}
  {'type': 'RECOGNISED_AS', 'count': 1}
  {'type': 'HAS_FATHER', 'count': 1}
  {'type': 'HAS_MOTHER', 'count': 1}
  {'type': 'HAS_BROTHER', 'count': 1}
  {'type': 'HAS_SISTER', 'count': 1}
  {'type': 'COUNTRY_OF', 'count': 1}
